In [ ]:
# Common imports for ESBE setup-style notebooks (1/2/3/9).
# Heavy lifting lives in urbanopt_des and lib.helpers; this cell stays terse.
import csv
import json
import shutil

from lib.helpers import (
    DEFAULT_DOCKER_IMAGE_TAG,
    select_uo_class,
    setup_notebook_paths,
)

# -- Execution mode ----------------------------------------------------------
# Set USE_DOCKER = True  to route uo commands through the Docker container.
# Set USE_DOCKER = False to use a locally installed URBANopt CLI.
USE_DOCKER = False
UO = select_uo_class(USE_DOCKER, DEFAULT_DOCKER_IMAGE_TAG)

# Autoreload dependencies while iterating on wrapper / helper code.
%load_ext autoreload
%autoreload 2

In [ ]:
# Standard set of working-directory paths used by every notebook.
# This notebook now targets the shared coincident project directory directly.
MODEL_OUTPUT_SUBDIR = "../coincident"
SCENARIO_NAME = "classproject_optimized"

paths = setup_notebook_paths(analysis_subdir=MODEL_OUTPUT_SUBDIR)
workdir = paths.workdir
analysis_dir = paths.analysis_dir
template_data_dir = paths.template_data_dir
num_usable_cores = paths.num_usable_cores

# Weather data is already present in the coincident project. No
# need to update the weather information.

### Activity 04a: Demand Flexibility


In [ ]:
# Use the existing coincident project directly from the configured root path.
coincident_dir = paths.analysis_dir
uo_coincident = UO(coincident_dir.parent, "coincident", template_dir=template_data_dir)

# Keep this alias so downstream cells continue to write outputs under the same root.
activity_04a_dir = uo_coincident.project_path

In [ ]:
# Verify the packaged ice storage flexibility measure is present in the project workflow.
workflow_path = activity_04a_dir / "mappers" / "base_workflow.osw"
with open(workflow_path) as f:
    workflow = json.load(f)

measure_names = [step.get("measure_dir_name") for step in workflow.get("steps", [])]
if "add_packaged_ice_storage" not in measure_names:
    raise RuntimeError("add_packaged_ice_storage is not present in base_workflow.osw")

print("Confirmed add_packaged_ice_storage is available in base_workflow.osw")

In [ ]:
# Create the flexibility mapper by copying ClassProject.rb and enabling AutoSize packaged ice storage.
mapper_dir = activity_04a_dir / "mappers"
source_mapper = mapper_dir / "ClassProject.rb"
flex_mapper = mapper_dir / "ClassProject_flex.rb"

if not flex_mapper.exists():
    shutil.copy2(source_mapper, flex_mapper)

mapper_text = flex_mapper.read_text(encoding="utf-8")
mapper_text = mapper_text.replace(
    "class ClassProjectMapper < BaselineMapper",
    "class FlexMapper < BaselineMapper",
)
mapper_text = mapper_text.replace(
    "OpenStudio::Extension.set_measure_argument(osw, 'add_packaged_ice_storage', '__SKIP__', true)",
    "OpenStudio::Extension.set_measure_argument(osw, 'add_packaged_ice_storage', '__SKIP__', false)",
)
flex_mapper.write_text(mapper_text, encoding="utf-8")

print(f"Wrote flexibility mapper: {flex_mapper}")

In [ ]:
# Create <SCENARIO_NAME>_flexibility_scenario.csv from <SCENARIO_NAME>.csv.
# Leave any buildings listed here on BaselineMapper if the flexibility measure is not applicable.
FLEX_EXCLUDED_FEATURES = []

base_scenario_name = f"{SCENARIO_NAME}.csv"
flex_scenario_name = f"{SCENARIO_NAME}_flexibility_scenario.csv"
baseline_scenario = activity_04a_dir / base_scenario_name
flex_scenario = activity_04a_dir / flex_scenario_name

with open(baseline_scenario, newline="") as f:
    rows = list(csv.DictReader(f))
fieldnames = rows[0].keys()

for row in rows:
    if row["Feature Name"] not in FLEX_EXCLUDED_FEATURES:
        row["Mapper Class"] = "URBANopt::Scenario::FlexMapper"

with open(flex_scenario, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote flexibility scenario: {flex_scenario}")

In [ ]:
# Create the REopt flexibility scenario mapper CSV from the flexibility baseline.
# This produces REopt_<SCENARIO_NAME>_flexibility_scenario.csv.
uo_coincident.create_reopt_scenario(
    "class_project_coincident.json",
    flex_scenario_name,
)
reopt_flex_scenario_name = f"REopt_{flex_scenario_name}"
print(f"Generated REopt scenario: {reopt_flex_scenario_name}")

In [ ]:
# Run the REopt baseline flexibility scenario.
uo_coincident.run(
    "class_project_coincident.json",
    reopt_flex_scenario_name,
)

In [ ]:
# Process and visualize the flexibility simulation results before REopt post-processing.
uo_coincident.process_scenario(
    "class_project_coincident.json",
    reopt_flex_scenario_name,
)
uo_coincident.visualize_feature("class_project_coincident.json")